<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 00 — Setup y verificación del entorno

**Proyecto:** Brecha digital y resultados Saber 11 — Grupo *REST pAPIs*
**Entrega 2:** Pipeline Bronze/Silver/Gold + MLlib sobre cluster Hadoop + Spark

Este notebook verifica que el SparkSession se conecta al cluster, que HDFS responde
y que se pueden leer los datos crudos. Es el "hello world" antes de la ingestión real.

## 1. Imports y SparkSession

In [1]:
import sys
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P, SPARK_MASTER, HDFS_NAMENODE
from pyspark.sql import functions as F

spark = get_spark("Entrega2-Setup")
sc = spark.sparkContext
print("Spark version :", spark.version)
print("Master        :", sc.master)
print("App name      :", sc.appName)
print("App ID        :", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)
print("Default FS    :", spark._jsc.hadoopConfiguration().get("fs.defaultFS"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/19 16:31:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/05/19 16:31:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version : 3.5.2
Master        : spark://spark-master:7077
App name      : Entrega2-Setup
App ID        : app-20260519163141-0018
Spark UI      : http://spark-master:4041
Default FS    : hdfs://10.43.97.164:9000


## 2. Recursos del cluster (workers vivos)

In [2]:
# StatusTracker en PySpark no expone executors; vamos por la JVM:
mem_status = sc._jsc.sc().getExecutorMemoryStatus()
print(f"Executors registrados: {mem_status.size()}")
it = mem_status.iterator()
while it.hasNext():
    entry = it.next()
    host = entry._1()
    remaining, max_mem = entry._2()._1(), entry._2()._2()
    print(f"  - {host}  free={remaining/1e9:.2f} GB  max={max_mem/1e9:.2f} GB")

print()
print("Default parallelism :", sc.defaultParallelism)
print("Default min parts   :", sc.defaultMinPartitions)

Executors registrados: 1
  - spark-master:36691  free=0.96 GB  max=0.96 GB

Default parallelism : 2
Default min parts   : 2


## 3. HDFS: contenido de `/usr/proyecto/bronze/`

In [3]:
# Usamos el FileSystem de Hadoop expuesto por la JVM de Spark
URI = sc._jvm.java.net.URI
Path = sc._jvm.org.apache.hadoop.fs.Path
FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem

fs = FileSystem.get(URI(HDFS_NAMENODE), sc._jsc.hadoopConfiguration())

def listdir(hdfs_dir):
    files = fs.listStatus(Path(hdfs_dir))
    return [(f.getPath().getName(), f.getLen(), f.isDirectory()) for f in files]

for sub in ["bronze", "bronze_parquet", "silver", "gold"]:
    p = f"/usr/proyecto/{sub}"
    try:
        items = listdir(p)
        print(f"\n=== {p} ===")
        for name, size, is_dir in items:
            tag = "DIR " if is_dir else "FILE"
            human = f"{size/1e6:>10.1f} MB" if not is_dir else " " * 12
            print(f"  {tag}  {human}  {name}")
    except Exception as e:
        print(f"\n=== {p} ===  (vacío o no existe)  {e}")


=== /usr/proyecto/bronze ===
  DIR                 icfes
  DIR                 internet
  DIR                 men
  DIR                 sisben

=== /usr/proyecto/bronze_parquet ===
  DIR                 icfes
  DIR                 internet
  DIR                 men
  DIR                 sisben

=== /usr/proyecto/silver ===
  DIR                 icfes
  DIR                 internet
  DIR                 men
  DIR                 sisben
  DIR                 sisben_municipal

=== /usr/proyecto/gold ===
  DIR                 clusters_municipales_k2_ano2022
  DIR                 icfes
  DIR                 internet
  DIR                 men
  DIR                 panel_municipal
  DIR                 sisben


## 4. Sanity check: leer MEN (5 MB) desde HDFS y mostrar cabeceras

In [4]:
# MEN es el más pequeño — buena verificación end-to-end sin gastar recursos.
df_men = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"hdfs://10.43.97.164:9000{P.BRONZE_CSV_MEN}")
)
print(f"MEN  ->  rows={df_men.count():,}   cols={len(df_men.columns)}")
df_men.printSchema()
df_men.show(3, truncate=40)

MEN  ->  rows=15,707   cols=41
root
 |-- AÑO: string (nullable = true)
 |-- CÓDIGO_MUNICIPIO: string (nullable = true)
 |-- MUNICIPIO: string (nullable = true)
 |-- CÓDIGO_DEPARTAMENTO: string (nullable = true)
 |-- DEPARTAMENTO: string (nullable = true)
 |-- CÓDIGO_ETC: string (nullable = true)
 |-- ETC: string (nullable = true)
 |-- POBLACIÓN_5_16: string (nullable = true)
 |-- TASA_MATRICULACIÓN_5_16: string (nullable = true)
 |-- COBERTURA_NETA: string (nullable = true)
 |-- COBERTURA_NETA_TRANSICIÓN: string (nullable = true)
 |-- COBERTURA_NETA_PRIMARIA: string (nullable = true)
 |-- COBERTURA_NETA_SECUNDARIA: string (nullable = true)
 |-- COBERTURA_NETA_MEDIA: string (nullable = true)
 |-- COBERTURA_BRUTA: string (nullable = true)
 |-- COBERTURA_BRUTA_TRANSICIÓN: string (nullable = true)
 |-- COBERTURA_BRUTA_PRIMARIA: string (nullable = true)
 |-- COBERTURA_BRUTA_SECUNDARIA: string (nullable = true)
 |-- COBERTURA_BRUTA_MEDIA: string (nullable = true)
 |-- TAMAÑO_PROMEDIO_DE_GRUP

26/05/19 16:31:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+----------------+---------+-------------------+------------+----------+---------------+--------------+-----------------------+--------------+-------------------------+-----------------------+-------------------------+--------------------+---------------+--------------------------+------------------------+--------------------------+---------------------+------------------------+---------------------------+---------+--------------------+------------------+--------------------+---------------+----------+---------------------+-------------------+---------------------+----------------+-----------+----------------------+--------------------+----------------------+-----------------+----------+---------------------+-------------------+---------------------+----------------+
| AÑO|CÓDIGO_MUNICIPIO|MUNICIPIO|CÓDIGO_DEPARTAMENTO|DEPARTAMENTO|CÓDIGO_ETC|            ETC|POBLACIÓN_5_16|TASA_MATRICULACIÓN_5_16|COBERTURA_NETA|COBERTURA_NETA_TRANSICIÓN|COBERTURA_NETA_PRIMARIA|COBERTURA_NETA_SECUN

## 5. Sanity check ICFES (head only, sin contar)

In [5]:
# Solo leemos schema + 5 filas (sin .count() para no cargar 3.5 GB).
df_icfes = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(f"hdfs://10.43.97.164:9000{P.BRONZE_CSV_ICFES}")
)
print(f"ICFES columnas: {len(df_icfes.columns)}")
print("Primeras 12 columnas:", df_icfes.columns[:12])
df_icfes.select("PERIODO", "COLE_COD_MCPIO_UBICACION", "COLE_DEPTO_UBICACION",
                "COLE_AREA_UBICACION", "FAMI_TIENEINTERNET", "PUNT_GLOBAL").show(5, truncate=False)

ICFES columnas: 51
Primeras 12 columnas: ['PERIODO', 'ESTU_TIPODOCUMENTO', 'ESTU_CONSECUTIVO', 'COLE_AREA_UBICACION', 'COLE_BILINGUE', 'COLE_CALENDARIO', 'COLE_CARACTER', 'COLE_COD_DANE_ESTABLECIMIENTO', 'COLE_COD_DANE_SEDE', 'COLE_COD_DEPTO_UBICACION', 'COLE_COD_MCPIO_UBICACION', 'COLE_CODIGO_ICFES']


+-------+------------------------+--------------------+-------------------+------------------+-----------+
|PERIODO|COLE_COD_MCPIO_UBICACION|COLE_DEPTO_UBICACION|COLE_AREA_UBICACION|FAMI_TIENEINTERNET|PUNT_GLOBAL|
+-------+------------------------+--------------------+-------------------+------------------+-----------+
|20131  |11001                   |BOGOTA              |URBANO             |Si                |NULL       |
|20194  |41016                   |HUILA               |RURAL              |Si                |339        |
|20194  |41016                   |HUILA               |RURAL              |Si                |339        |
|20122  |63130                   |QUINDIO             |URBANO             |Si                |NULL       |
|20132  |19001                   |CAUCA               |URBANO             |Si                |NULL       |
+-------+------------------------+--------------------+-------------------+------------------+-----------+
only showing top 5 rows



## 6. Conclusión

Si todas las celdas anteriores ejecutaron sin error:

- ✅ El SparkSession se conecta al cluster `spark://spark-master:7077`.
- ✅ HDFS responde en `hdfs://10.43.97.164:9000` y los CSVs Bronze están allí.
- ✅ Spark puede leer CSV desde HDFS respetando UTF-8.

**Siguiente paso:** notebook `01_bronze_csv_a_parquet.ipynb` — convierte los 4 CSVs a Parquet snappy.

In [6]:
spark.stop()

26/05/19 16:32:04 WARN NioEventLoop: Selector.select() returned prematurely 512 times in a row; rebuilding Selector io.netty.channel.nio.SelectedSelectionKeySetSelector@4754e972.
